# Week 1 — Exploration

**Where to run:** Google Colab (free tier — 12.7 GB RAM, 100 GB disk). Open via *File → Upload notebook* or *File → Open notebook → GitHub*.

**Universe / files** (from `description.md`):

| table | exchange | files | columns |
|---|---|---|---|
| trades        | binance | `binance_trades/perp_{btc,eth}usdt.parquet`       | `timestamp, ticker, side, price, amount` |
| booktickers   | binance | `binance_booktickers/perp_{btc,eth}usdt.parquet`  | `timestamp, ticker, bid_price, bid_amount, ask_price, ask_amount` |
| liquidations  | binance | `binance_liquidations/perp_{btc,eth}usdt.parquet` | `timestamp, ticker, side, price, amount` |
| liquidations  | bybit   | `bybit_liquidations/{btc,eth}usdt.parquet`        | `timestamp, ticker, side, price, amount` |

**Conventions:**
- `timestamp` = `int64`, **µs since UNIX epoch UTC**, single time axis everywhere.
- `side` in **trades** = *taker* side. `buy` ⇒ taker bought ⇒ maker sold.
- `side` in **liquidations** = side of the *liquidation order*. `buy` ⇒ short forcibly closed ⇒ upward pressure.
- **Bybit** events are visible to us only after **200 ms** network delay.

**End-goal of the project** (later weeks): for each Binance trade, decide `keep` / `filter` so the maker side of kept trades has positive markout PnL at horizons 30/120/300 s. This week: *look at the data*.

**RAM discipline kept throughout** even though Colab has 12 GB: streaming groupbys; downsampled quantiles; time-window slicing for cross-source analyses. Biggest table is trades-ETH = 1.37 B rows.

Sections: setup · inventory · smoke · shape · gaps · trade distributions · BBO distributions · liquidation distributions · BBO around trade · BBO around liquidation · conventions · Bybit delay · cross-exchange alignment · findings log.

## 1. Setup

Auto-detects Colab vs local.

- **In Colab**: installs `polars` + `pyarrow` if missing, downloads the source archive via `gdown` (5-10 min the first time of the session), untars into `/content/`.
- **Locally**: assumes data already unpacked into `./liquidation_task/data/`.

In [ ]:
import os, sys, gc, subprocess, shutil
from pathlib import Path
from datetime import datetime, timezone

IN_COLAB = "google.colab" in sys.modules
print("running in:", "COLAB" if IN_COLAB else "LOCAL")

# Source archive on Google Drive ("anyone with link" sharing).
FILE_ID = "1XmxRsElei-vE8Gc5tkKs2wH4FJVRTevS"   # liquidation_task_0520.tar  (~10 GB)

if IN_COLAB:
    # Ensure polars + pyarrow.
    for pkg in ("polars", "pyarrow"):
        try:
            __import__(pkg)
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
    # gdown is preinstalled in Colab.
    TAR_PATH = Path("/content/liquidation_task.tar")
    ROOT     = Path("/content/liquidation_task/data")
    if not ROOT.exists():
        if not TAR_PATH.exists():
            print("downloading tar via gdown — this is the slow step (~5-10 min on a good Colab node)")
            subprocess.run(["gdown", "--id", FILE_ID, "-O", str(TAR_PATH)], check=True)
        print("extracting tar to /content/ …")
        subprocess.run(["tar", "-xf", str(TAR_PATH), "-C", "/content/"], check=True)
        print("removing tar to free disk")
        TAR_PATH.unlink(missing_ok=True)
    else:
        print(f"data already extracted at {ROOT}")
else:
    ROOT = Path("liquidation_task/data")

assert ROOT.exists(), f"ROOT not found: {ROOT.resolve()}"
print(f"ROOT = {ROOT}")
print("size:", sum(p.stat().st_size for p in ROOT.rglob('*') if p.is_file()) // 1024**2, "MB")

In [ ]:
import numpy as np
import polars as pl
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(20)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 110

FILES = {
    ("binance", "trades",       "BTC"): ROOT / "binance_trades"       / "perp_btcusdt.parquet",
    ("binance", "trades",       "ETH"): ROOT / "binance_trades"       / "perp_ethusdt.parquet",
    ("binance", "bbo",          "BTC"): ROOT / "binance_booktickers"  / "perp_btcusdt.parquet",
    ("binance", "bbo",          "ETH"): ROOT / "binance_booktickers"  / "perp_ethusdt.parquet",
    ("binance", "liquidations", "BTC"): ROOT / "binance_liquidations" / "perp_btcusdt.parquet",
    ("binance", "liquidations", "ETH"): ROOT / "binance_liquidations" / "perp_ethusdt.parquet",
    ("bybit",   "liquidations", "BTC"): ROOT / "bybit_liquidations"   / "btcusdt.parquet",
    ("bybit",   "liquidations", "ETH"): ROOT / "bybit_liquidations"   / "ethusdt.parquet",
}
for k, p in FILES.items():
    assert p.exists(), f"missing {k}: {p}"
print("all 8 files present")

In [ ]:
def scan(exchange, table, symbol) -> pl.LazyFrame:
    return pl.scan_parquet(FILES[(exchange, table, symbol)])

def load_small(exchange, table, symbol) -> pl.DataFrame:
    assert table == "liquidations", f"refuse to load whole {table} table"
    return pl.read_parquet(FILES[(exchange, table, symbol)])

def n_rows(exchange, table, symbol) -> int:
    return pq.ParquetFile(FILES[(exchange, table, symbol)]).metadata.num_rows

def ts_range(exchange, table, symbol) -> tuple[int, int]:
    """Min/max timestamp in µs via parquet row-group statistics."""
    pf = pq.ParquetFile(FILES[(exchange, table, symbol)])
    md = pf.metadata
    idx = pf.schema_arrow.get_field_index("timestamp")
    mins, maxs = [], []
    for i in range(md.num_row_groups):
        st = md.row_group(i).column(idx).statistics
        if st is not None and st.has_min_max:
            mins.append(st.min); maxs.append(st.max)
    return min(mins), max(maxs)

def fmt_ts(ts_us: int) -> str:
    return datetime.fromtimestamp(ts_us / 1e6, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")

def sample_lazy(lf: pl.LazyFrame, n_total: int, target: int) -> pl.LazyFrame:
    step = max(1, n_total // target)
    return lf.gather_every(step)

def window_lf(lf: pl.LazyFrame, t_min: int, t_max: int) -> pl.LazyFrame:
    return lf.filter((pl.col("timestamp") >= t_min) & (pl.col("timestamp") <= t_max))

## 2. Inventory + schemas

All numbers via parquet footers — zero data is read.

In [ ]:
rows = []
for (ex, tbl, sym), p in FILES.items():
    n  = n_rows(ex, tbl, sym)
    tmin, tmax = ts_range(ex, tbl, sym)
    days = (tmax - tmin) / 86_400_000_000
    rows.append({
        "exch": ex, "tbl": tbl, "sym": sym,
        "file_MB": round(p.stat().st_size / 1024**2, 1),
        "rows": n,
        "from": fmt_ts(tmin)[:16],
        "to":   fmt_ts(tmax)[:16],
        "days": round(days, 2),
        "rows_per_day": int(n / max(days, 1)),
    })
INV = pl.DataFrame(rows)
INV

In [ ]:
for (ex, tbl, sym), p in FILES.items():
    schema = pq.ParquetFile(p).schema_arrow
    print(f"{ex:8} {tbl:13} {sym:4} :: " + ", ".join(f"{f.name}:{f.type}" for f in schema))

## 3. Smoke — head + tail

In [ ]:
for (ex, tbl, sym) in FILES.keys():
    lf = scan(ex, tbl, sym)
    h = lf.head(3).collect()
    t = lf.tail(3).collect()
    print(f"\n=== {ex} / {tbl} / {sym} ===")
    print("head:"); print(h)
    print("tail:"); print(t)

## 4. Shape — per-day and intra-day

In [ ]:
def daily_counts(ex, tbl, sym) -> pl.DataFrame:
    return (
        scan(ex, tbl, sym)
          .with_columns((pl.col("timestamp") // 86_400_000_000).alias("day_int"))
          .group_by("day_int").agg(pl.len().alias("n"))
          .sort("day_int")
          .collect(engine="streaming")
          .with_columns(pl.from_epoch(pl.col("day_int") * 86_400, time_unit="s").alias("day"))
    )

fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)
for ax, (ex, tbl, sym) in zip(axes.flat, FILES.keys()):
    d = daily_counts(ex, tbl, sym)
    ax.bar(d["day"].to_numpy(), d["n"].to_numpy(), width=0.9)
    ax.set_title(f"{ex} {tbl} {sym} — {d['n'].sum():,} rows / {len(d)} days")
    ax.set_ylabel("rows/day")
    for label in ax.get_xticklabels(): label.set_rotation(45)
fig.tight_layout(); plt.show()

In [ ]:
def hourly_counts(ex, tbl, sym) -> pl.DataFrame:
    return (
        scan(ex, tbl, sym)
          .with_columns(((pl.col("timestamp") // 3_600_000_000) % 24).alias("hour"))
          .group_by("hour").agg(pl.len().alias("n"))
          .sort("hour")
          .collect(engine="streaming")
    )

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
for ax, (ex, tbl) in zip(axes.flat, [("binance","trades"), ("binance","bbo"),
                                       ("binance","liquidations"), ("bybit","liquidations")]):
    for sym in ("BTC", "ETH"):
        d = hourly_counts(ex, tbl, sym)
        ax.plot(d["hour"].to_numpy(), d["n"].to_numpy(), marker="o", label=sym)
    ax.set_title(f"{ex} {tbl} — events per hour-of-day (UTC)")
    ax.set_xlabel("hour UTC"); ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); plt.show()

## 5. Gaps & duplicates

In [ ]:
# Streaming aggregation; cheap.
print(f"{'exch':8} {'tbl':13} {'sym':4} {'rows':>14} {'unique_ts':>14} {'ratio':>8}")
for (ex, tbl, sym) in FILES:
    n_total = n_rows(ex, tbl, sym)
    n_uniq = scan(ex, tbl, sym).select(pl.col("timestamp").n_unique()).collect(engine="streaming").item()
    print(f"{ex:8} {tbl:13} {sym:4} {n_total:>14,} {n_uniq:>14,} {n_uniq/n_total:>8.3f}")

In [ ]:
# Δt percentiles via a contiguous HEAD sample (10 M rows). Δt needs *adjacent* rows.
HEAD_N = 10_000_000
print(f"{'exch':8} {'tbl':13} {'sym':4} {'sample':>11}   p50ms     p90ms     p99ms    p999ms      maxms")
for (ex, tbl, sym) in FILES:
    n_total = n_rows(ex, tbl, sym)
    n_take  = min(HEAD_N, n_total)
    ts = scan(ex, tbl, sym).head(n_take).select("timestamp").collect()["timestamp"].to_numpy()
    if not (np.diff(ts) >= 0).all():
        ts = np.sort(ts)
    dt_ms = np.diff(ts) / 1_000.0
    p50, p90, p99, p999, mx = np.percentile(dt_ms, [50, 90, 99, 99.9, 100])
    print(f"{ex:8} {tbl:13} {sym:4} {len(dt_ms):>11,} {p50:>9.3f} {p90:>9.3f} {p99:>9.3f} {p999:>10.3f} {mx:>10.1f}")
    del ts, dt_ms; gc.collect()

## 6. Distributions: trades

In [ ]:
for sym in ("BTC", "ETH"):
    n_total = n_rows("binance", "trades", sym)
    exact = scan("binance", "trades", sym).select([
        pl.col("price").min().alias("price_min"),
        pl.col("price").max().alias("price_max"),
        pl.col("price").mean().alias("price_mean"),
        pl.col("amount").min().alias("amt_min"),
        pl.col("amount").max().alias("amt_max"),
        (pl.col("price") * pl.col("amount")).sum().alias("sum_notional_USD"),
        ((pl.col("price") * pl.col("amount")) > 100_000).mean().alias("frac_above_clip_100k"),
    ]).collect(engine="streaming")
    sample = (
        sample_lazy(scan("binance", "trades", sym), n_total, target=1_000_000)
        .with_columns((pl.col("price") * pl.col("amount")).alias("notional"))
        .select("notional")
        .collect(engine="streaming")["notional"].to_numpy()
    )
    p50, p90, p99, p999, p9999 = np.percentile(sample, [50, 90, 99, 99.9, 99.99])
    print(f"\n=== binance trades {sym} (n={n_total:,}) ===")
    print(exact)
    print(f"notional USD (sampled n={len(sample):,}): p50={p50:,.0f}  p90={p90:,.0f}  p99={p99:,.0f}  p99.9={p999:,.0f}  p99.99={p9999:,.0f}")
    side = scan("binance", "trades", sym).group_by("side").agg([
        pl.len().alias("n"),
        (pl.col("price") * pl.col("amount")).sum().alias("sum_notional_USD"),
    ]).sort("side").collect(engine="streaming")
    print(side)
    del sample; gc.collect()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, sym in zip(axes, ("BTC", "ETH")):
    n_total = n_rows("binance", "trades", sym)
    sample = (
        sample_lazy(scan("binance", "trades", sym), n_total, target=1_000_000)
        .with_columns((pl.col("price") * pl.col("amount")).alias("n"))
        .filter(pl.col("n") > 0)
        .select("n")
        .collect(engine="streaming")["n"].to_numpy()
    )
    ax.hist(np.log10(sample), bins=80)
    ax.axvline(np.log10(100_000), color="r", lw=1, label="clip @ 100k")
    ax.set_title(f"binance trades {sym} — log10(notional USD), n={len(sample):,}")
    ax.legend()
fig.tight_layout(); plt.show()
del sample; gc.collect()

## 7. Distributions: BBO — spread, sizes

In [ ]:
for sym in ("BTC", "ETH"):
    n_total = n_rows("binance", "bbo", sym)
    exact = scan("binance", "bbo", sym).with_columns([
        ((pl.col("ask_price") - pl.col("bid_price")) /
         ((pl.col("ask_price") + pl.col("bid_price")) / 2) * 1e4).alias("spread_bps"),
    ]).select([
        pl.col("spread_bps").min().alias("sp_min"),
        pl.col("spread_bps").max().alias("sp_max"),
        (pl.col("spread_bps") < 0).sum().alias("n_crossed"),
        (pl.col("spread_bps") == 0).sum().alias("n_locked"),
    ]).collect(engine="streaming")
    sample = (
        sample_lazy(scan("binance", "bbo", sym), n_total, target=1_000_000)
        .select([
            ((pl.col("ask_price") - pl.col("bid_price")) /
             ((pl.col("ask_price") + pl.col("bid_price")) / 2) * 1e4).alias("s"),
            pl.col("bid_amount").alias("ba"),
            pl.col("ask_amount").alias("aa"),
        ])
        .collect(engine="streaming")
    )
    s = sample["s"].to_numpy()
    sp50, sp90, sp99 = np.percentile(s, [50, 90, 99])
    bsz = np.median(sample["ba"].to_numpy())
    asz = np.median(sample["aa"].to_numpy())
    print(f"\n=== binance bbo {sym} (n={n_total:,}) ===")
    print(exact)
    print(f"spread bps (sampled n={len(s):,}): p50={sp50:.3f}  p90={sp90:.3f}  p99={sp99:.3f}")
    print(f"median bid size = {bsz:.3f}   median ask size = {asz:.3f}")
    del s, sample; gc.collect()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, sym in zip(axes, ("BTC", "ETH")):
    n_total = n_rows("binance", "bbo", sym)
    sample = (
        sample_lazy(scan("binance", "bbo", sym), n_total, target=1_000_000)
        .select(((pl.col("ask_price") - pl.col("bid_price")) /
                 ((pl.col("ask_price") + pl.col("bid_price")) / 2) * 1e4).alias("s"))
        .filter(pl.col("s") > 0)
        .collect(engine="streaming")["s"].to_numpy()
    )
    ax.hist(np.log10(sample), bins=80)
    ax.set_title(f"binance bbo {sym} — log10(spread_bps), n={len(sample):,}")
fig.tight_layout(); plt.show()
del sample; gc.collect()

## 8. Distributions: liquidations

In [ ]:
liqs = {}
for ex in ("binance", "bybit"):
    for sym in ("BTC", "ETH"):
        df = load_small(ex, "liquidations", sym).with_columns(
            (pl.col("price") * pl.col("amount")).alias("notional")
        )
        liqs[(ex, sym)] = df
        by_side = df.group_by("side").agg([
            pl.len().alias("n"),
            pl.col("notional").sum().alias("sum_notional"),
            pl.col("notional").median().alias("med_notional"),
            pl.col("notional").max().alias("max_notional"),
        ]).sort("side")
        print(f"\n=== {ex} liquidations {sym} (n={len(df):,}) ===")
        print(by_side)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, ((ex, sym), df) in zip(axes.flat, liqs.items()):
    n = df.filter(pl.col("notional") > 0)["notional"].to_numpy()
    ax.hist(np.log10(n), bins=60)
    ax.set_title(f"{ex} liq {sym} (n={len(n):,})  log10(notional USD)")
fig.tight_layout(); plt.show()

## 9. BBO around a Binance trade (1-hour window)

In [ ]:
sym = "BTC"
WINDOW_H = 1
tmin, tmax = ts_range("binance", "trades", sym)
t0  = (tmin + tmax) // 2
t1  = t0 + WINDOW_H * 3_600_000_000
pad = 5_000_000  # 5 s pad for asof safety

tr  = window_lf(scan("binance", "trades", sym), t0, t1)\
        .select(["timestamp","side","price","amount"]).sort("timestamp").collect()
bbo = window_lf(scan("binance", "bbo",    sym), t0 - pad, t1 + pad)\
        .select(["timestamp","bid_price","ask_price","bid_amount","ask_amount"]).sort("timestamp").collect()
print(f"window {fmt_ts(t0)} — {fmt_ts(t1)}  ({WINDOW_H}h)")
print(f"  trades: {len(tr):>10,}")
print(f"  bbo:    {len(bbo):>10,}")

merged = tr.join_asof(bbo, on="timestamp", strategy="backward").with_columns([
    ((pl.col("bid_price") + pl.col("ask_price")) / 2).alias("mid"),
    (pl.col("price") - (pl.col("bid_price") + pl.col("ask_price")) / 2).alias("trade_minus_mid"),
])
merged.head(8)

In [ ]:
# `side` taker check:  buy → at ask (price >= mid),  sell → at bid (price <= mid)
side_check = (
    merged.with_columns((pl.col("price") > pl.col("mid")).alias("above_mid"))
          .group_by("side")
          .agg([
              pl.len().alias("n"),
              pl.col("above_mid").mean().alias("share_above_mid"),
              ((pl.col("price") - pl.col("mid")).mean()).alias("avg_dev_from_mid"),
          ])
          .sort("side")
)
side_check

## 10. BBO around a Binance liquidation (chunked windows)

In [ ]:
def mids_around_events(events_ts_us: np.ndarray, sym: str, offsets_ms: list[int],
                       chunk_size: int = 2_000, max_pad_us: int = 5_000_000) -> np.ndarray:
    """Returns shape (n_events, n_offsets) with Binance BBO mid at each event_ts + o*ms."""
    events_ts_us = np.sort(events_ts_us)
    n_events  = len(events_ts_us)
    n_offsets = len(offsets_ms)
    max_off_us = max(abs(o) for o in offsets_ms) * 1_000 + max_pad_us
    out = np.full((n_events, n_offsets), np.nan)
    for start in range(0, n_events, chunk_size):
        end = min(start + chunk_size, n_events)
        chunk_ts = events_ts_us[start:end]
        t_lo = int(chunk_ts.min()) - max_off_us
        t_hi = int(chunk_ts.max()) + max_off_us
        bbo_w = (
            window_lf(scan("binance", "bbo", sym), t_lo, t_hi)
              .select(["timestamp","bid_price","ask_price"])
              .sort("timestamp")
              .collect()
        )
        if bbo_w.is_empty(): continue
        ts_arr  = bbo_w["timestamp"].to_numpy()
        mid_arr = (bbo_w["bid_price"].to_numpy() + bbo_w["ask_price"].to_numpy()) * 0.5
        for j, o in enumerate(offsets_ms):
            target = chunk_ts + o * 1_000
            idx = np.searchsorted(ts_arr, target, side="right") - 1
            valid = (idx >= 0) & (target >= ts_arr[0])
            out[start:end, j] = np.where(valid, mid_arr[np.clip(idx, 0, len(ts_arr)-1)], np.nan)
        del bbo_w, ts_arr, mid_arr
    return out

In [ ]:
sym = "BTC"
liq = load_small("binance", "liquidations", sym).sort("timestamp")
offsets_ms = [-100, 0, 100, 1_000, 10_000, 60_000]

mids = mids_around_events(liq["timestamp"].to_numpy(), sym, offsets_ms, chunk_size=2000)
side = liq["side"].to_numpy()
base = mids[:, offsets_ms.index(0)]

for sd in ("buy", "sell"):
    mask = side == sd
    print(f"\nside={sd}  (n={mask.sum():,})")
    for j, o in enumerate(offsets_ms):
        dev = (mids[mask, j] - base[mask]) / base[mask] * 1e4
        print(f"  t{o:+6d}ms  median dev = {np.nanmedian(dev):+7.3f} bps   p25/p75 = {np.nanpercentile(dev,25):+7.3f} / {np.nanpercentile(dev,75):+7.3f}")

## 11. Conventions in practice

In [ ]:
for (ex, tbl, sym) in FILES:
    one = scan(ex, tbl, sym).select(pl.col("timestamp")).head(1).collect().item()
    mag = int(np.log10(one))
    guess = {12: "ms", 15: "µs", 18: "ns"}.get(mag, f"10^{mag}?")
    print(f"{ex:8} {tbl:13} {sym:4}  ts={one}  → unit guess: {guess}  → {fmt_ts(one)}")

## 12. Bybit cross-exchange delay

Claim: Bybit events become available +200 ms after their timestamp. Test: for each Bybit liquidation, compute Binance mid at offsets −1 s … +2 s, sign by side (buy=+1, sell=−1), take the median across all events at each offset. Peak of the curve ≈ effective delay.

In [ ]:
sym = "BTC"
by_liq = load_small("bybit", "liquidations", sym).sort("timestamp")
offsets_ms = list(range(-1000, 2001, 50))

mids = mids_around_events(by_liq["timestamp"].to_numpy(), sym, offsets_ms, chunk_size=2000)
sign = np.where(by_liq["side"].to_numpy() == "buy", +1.0, -1.0)
base = mids[:, offsets_ms.index(0)]

response = np.array([
    np.nanmedian((mids[:, j] - base) / base * 1e4 * sign)
    for j in range(len(offsets_ms))
])

plt.figure(figsize=(10, 4))
plt.plot(offsets_ms, response, marker=".")
plt.axvline(200, color="r", ls="--", label="claimed delay 200 ms")
plt.axhline(0, color="k", lw=0.5)
plt.xlabel("offset bybit→binance (ms)")
plt.ylabel("signed median mid move (bps)")
plt.title(f"Binance mid response around Bybit liquidations ({sym}, n={len(by_liq):,})")
plt.legend(); plt.grid(alpha=0.3); plt.show()

peak = int(offsets_ms[int(np.nanargmax(response))])
print(f"peak signed response at offset = {peak} ms")

## 13. Cross-exchange liquidation alignment

In [ ]:
for sym in ("BTC", "ETH"):
    a = load_small("binance", "liquidations", sym).select(["timestamp","side"]).sort("timestamp")
    b = load_small("bybit",   "liquidations", sym).select(["timestamp","side"]).sort("timestamp")
    j = a.join_asof(
        b.rename({"timestamp":"ts_by","side":"side_by"}),
        left_on="timestamp", right_on="ts_by",
        strategy="nearest", tolerance=2_000_000,
    )
    matched = j.filter(pl.col("ts_by").is_not_null() & (pl.col("side") == pl.col("side_by")))
    dt_ms = ((matched["ts_by"] - matched["timestamp"]) / 1_000).to_numpy()
    print(f"{sym}: {len(matched):,}/{len(a):,} binance liqs matched a same-side bybit liq within ±2s")
    if len(dt_ms):
        print(f"     median Δt = {np.median(dt_ms):.1f} ms,  p10/p90 = {np.percentile(dt_ms,10):.1f} / {np.percentile(dt_ms,90):.1f} ms")
        plt.figure(figsize=(8,3))
        plt.hist(dt_ms, bins=80)
        plt.axvline(0, color="k", lw=0.5)
        plt.title(f"{sym} same-side Δt (bybit minus binance, ms)")
        plt.show()

---
## 14. Findings log

Fill in numbers from the cells above:

- date range covered (UTC): 
- rows/day, trades BTC vs ETH: 
- intra-day pattern (UTC hours of peak activity): 
- timestamp unit (verified µs): ✓ / ✗
- trades.side semantics (taker, verified): ✓ / ✗
- liquidation.side semantics (buy ⇒ up, verified): ✓ / ✗
- Bybit→Binance delay (measured peak offset, ms): 
- median spread BTC / ETH (bps): 
- biggest single liquidation seen (notional USD): 
- anything weird: 
